# Полное сравнение всех методов unfold_

Этот ноутбук проводит комплексное сравнение всех доступных методов восстановления спектра в bssunfold.

## Цели:
1. Перебрать все параметры каждого метода
2. Составить таблицу эффективности восстановления спектра
3. Выдать отчет по методам с метриками
4. Построить графики спектров

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
from itertools import product

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
print("Libraries imported")

In [ ]:
import sys
sys.path.insert(0, '/workspace/src')
from bssunfold.core import (
    solve_mlem, solve_maxed, solve_tsvd, solve_bayes,
    solve_cvxpy, solve_landweber, solve_sart, solve_kaczmarz,
    solve_cgls, solve_lanczos, solve_tikhonov_tv, solve_gravel,
    solve_doroshenko, solve_bunki, solve_sandii, solve_osem,
    solve_qpsolvers, solve_statreg, solve_reconst,
    solve_scipy_direct, solve_gks, solve_mapem, solve_bsrem,
    solve_ferdor, solve_rebunki, solve_bunkiut,
    solve_lmfit, solve_genetic, solve_mystic,
    solve_tikhonov_legendre, solve_bayes_spline,
    solve_cs, solve_omp, solve_sl0
)
print(f"Imported {33} unfold methods")

In [ ]:
df_spectra = pd.read_csv('/workspace/tests/MonteCarlo_Calculated_spectra_from_IAEA_Comp_for_comparison.csv')
print(f"Loaded {len(df_spectra.columns) - 1} spectra")
energy = df_spectra['E_MeV'].values

In [ ]:
def calculate_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / (ss_tot + 1e-30))

def calculate_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def calculate_mape(y_true, y_pred):
    mask = y_true > 0
    if not np.any(mask): return np.inf
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

METRICS = {'R2': calculate_r2, 'Pearson': lambda t,p: np.corrcoef(t,p)[0,1], 'RMSE': calculate_rmse, 'MAPE_pct': calculate_mape}
print("Metrics defined")

In [ ]:
METHODS = {
    'mlem': {'func': solve_mlem, 'params': {'iterations': [10, 50]}},
    'maxed': {'func': solve_maxed, 'params': {}},
    'tsvd': {'func': solve_tsvd, 'params': {'n_components': [5, 10]}},
    'bayes': {'func': solve_bayes, 'params': {}},
    'landweber': {'func': solve_landweber, 'params': {'iterations': [50], 'alpha': [0.01]}},
    'sart': {'func': solve_sart, 'params': {'iterations': [10]}},
    'kaczmarz': {'func': solve_kaczmarz, 'params': {'iterations': [50]}},
    'cgls': {'func': solve_cgls, 'params': {'iterations': [20]}},
    'gravel': {'func': solve_gravel, 'params': {}},
    'doroshenko': {'func': solve_doroshenko, 'params': {}},
    'bunki': {'func': solve_bunki, 'params': {}},
    'sandii': {'func': solve_sandii, 'params': {}},
    'qpsolvers': {'func': solve_qpsolvers, 'params': {}},
    'scipy_direct': {'func': solve_scipy_direct, 'params': {}},
}
print(f"Configured {len(METHODS)} methods")

In [ ]:
np.random.seed(42)
n_bins = len(energy)
true_spectrum = df_spectra['ISO_ref_Cf252'].values
response = np.diag(0.5 + 0.3*np.random.random(n_bins))
for i in range(n_bins-1): response[i,i+1] = 0.1*np.random.random()
counts = np.maximum(response @ true_spectrum + 0.01*np.random.randn(n_bins), 0)
print(f"Test setup complete: {n_bins} bins")

In [ ]:
results = []
for name, cfg in METHODS.items():
    params_list = [dict(zip(cfg['params'].keys(), v)) for v in product(*cfg['params'].values())] if cfg['params'] else [{}]
    for params in params_list:
        try:
            t0 = time.time()
            res = cfg['func'](counts, response, energy, **params)
            elapsed = time.time() - t0
            unfolded = res.get('unfolded') if isinstance(res, dict) else res
            if unfolded is None:
                for k,v in (res.items() if isinstance(res,dict) else []):
                    if isinstance(v, np.ndarray) and v.shape == energy.shape: unfolded = v; break
            if unfolded is not None and len(unfolded) == len(energy):
                metrics = {k: float(v(true_spectrum, unfolded)) for k,v in METRICS.items()}
                results.append({'method': name, 'params': str(params), 'time': elapsed, 'unfolded': unfolded, **metrics})
                print(f"{name}: R2={metrics['R2']:.3f}, time={elapsed:.2f}s")
        except Exception as e:
            print(f"{name} failed: {e}")
print(f"\nCompleted {len(results)} runs")

In [ ]:
if results:
    df = pd.DataFrame(results).drop(columns=['unfolded'])
    ranking = df.groupby('method')[['R2','Pearson','RMSE','time']].mean().sort_values('R2', ascending=False)
    print(ranking.to_string())
    ranking.to_csv('/workspace/examples/unfold_ranking.csv')
    print("\nSaved ranking to unfold_ranking.csv")

In [ ]:
if results:
    df = pd.DataFrame(results)
    plt.figure(figsize=(10,6))
    agg = df.groupby('method')['R2'].mean().sort_values(ascending=True).tail(10)
    plt.barh(agg.index, agg.values)
    plt.xlabel('R2 Score')
    plt.title('Top 10 Methods by R2')
    plt.tight_layout()
    plt.savefig('/workspace/examples/unfold_comparison.png', dpi=150)
    print("Saved plot to unfold_comparison.png")
    plt.show()